# 04 Transformers: XLM-R vs AfroXLMR (E5)

`python -m src.train_transformer --model xlmr afroxlmr --seeds 13 42 2026`

**Data flow:** SMS -> SentencePiece sub-words (max 128) -> 12-layer encoder -> `<s>` vector -> dense + tanh -> linear -> softmax -> P(scam).

**Settings:** AdamW, lr 3e-5, weight decay 0.01, 10% warm-up, batch 16, up to 5 epochs, early stopping on validation F1, class-weighted cross-entropy.

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('MPLCONFIGDIR', os.path.abspath('../.cache/mpl'))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option('display.max_colwidth', 140)
pd.set_option('display.width', 200)
from src import config
S = pd.read_csv(config.RESULTS / 'experiments.csv')          # mean (and sd) over seeds
BY_SEED = pd.read_csv(config.RESULTS / 'experiments_by_seed.csv')

def table(models, sets, variant='clean', split='template', metric='f1'):
    t = S[S.model.isin(models) & S.set.isin(sets) & (S.variant == variant) & (S.split == split)]
    return t.pivot_table(index='model', columns='set', values=metric).reindex(models).round(3)

In [2]:
from transformers import AutoTokenizer
for name in config.TRANSFORMERS.values():
    tok = AutoTokenizer.from_pretrained(name)
    print(name, tok.tokenize('Iyo pesa itume kwenye namba hii <PHONE> jina litakuja JUMA'))

FacebookAI/xlm-roberta-base ['▁Iyo', '▁pesa', '▁itu', 'me', '▁kwenye', '▁namba', '▁hii', '▁<', 'PHONE', '>', '▁jina', '▁lita', 'kuja', '▁J', 'UMA']
Davlan/afro-xlmr-base ['▁Iyo', '▁pesa', '▁itu', 'me', '▁kwenye', '▁namba', '▁hii', '▁<', 'PHONE', '>', '▁jina', '▁lita', 'kuja', '▁J', 'UMA']


## Training curves

In [3]:
log = pd.read_json(config.RESULTS / 'transformer_log.jsonl', lines=True)
log[['model', 'variant', 'seed', 'lr', 'best_val_f1', 'seconds', 'history']]

,model,variant,seed,lr,best_val_f1,seconds,history
0,afroxlmr,clean,42,0.00003,1.000000,755,"[{'epoch': 1, 'val_f1': 1.0, 'val_loss': 0.0013000000000000002}, {'epoch': 2, 'val_f1': 1.0, 'val_loss': 0.0006000000000000001}, {'epoch..."
1,xlmr,clean,42,0.00003,1.000000,781,"[{'epoch': 1, 'val_f1': 1.0, 'val_loss': 0.0031000000000000003}, {'epoch': 2, 'val_f1': 1.0, 'val_loss': 0.0004}, {'epoch': 3, 'val_f1':..."
2,afroxlmr,clean,13,0.00003,0.993865,430,"[{'epoch': 1, 'val_f1': 0.9939, 'val_loss': 0.0196}, {'epoch': 2, 'val_f1': 0.9939, 'val_loss': 0.0408}, {'epoch': 3, 'val_f1': 0.9939, ..."
3,afroxlmr,clean,2026,0.00003,1.000000,532,"[{'epoch': 1, 'val_f1': 1.0, 'val_loss': 0.0035}, {'epoch': 2, 'val_f1': 0.9939, 'val_loss': 0.0408}, {'epoch': 3, 'val_f1': 0.9939, 'va..."
4,xlmr,clean,13,0.00003,1.000000,716,"[{'epoch': 1, 'val_f1': 1.0, 'val_loss': 0.0106}, {'epoch': 2, 'val_f1': 1.0, 'val_loss': 0.0001}, {'epoch': 3, 'val_f1': 0.9939, 'val_l..."
5,xlmr,clean,2026,0.00003,0.993865,857,"[{'epoch': 1, 'val_f1': 0.9939, 'val_loss': 0.0596}, {'epoch': 2, 'val_f1': 0.9939, 'val_loss': 0.0356}, {'epoch': 3, 'val_f1': 0.9939, ..."
6,afroxlmr,advtrain,13,0.00003,0.993865,620,"[{'epoch': 1, 'val_f1': 0.9939, 'val_loss': 0.019200000000000002}, {'epoch': 2, 'val_f1': 0.9939, 'val_loss': 0.038400000000000004}, {'e..."
7,afroxlmr,advtrain,42,0.00003,1.000000,741,"[{'epoch': 1, 'val_f1': 1.0, 'val_loss': 0.0025}, {'epoch': 2, 'val_f1': 1.0, 'val_loss': 0.0021000000000000003}, {'epoch': 3, 'val_f1':..."
8,afroxlmr,advtrain,2026,0.00003,1.000000,546,"[{'epoch': 1, 'val_f1': 1.0, 'val_loss': 0.001}, {'epoch': 2, 'val_f1': 0.9939, 'val_loss': 0.029500000000000002}, {'epoch': 3, 'val_f1'..."
9,afroxlmr,strip,13,0.00003,1.000000,693,"[{'epoch': 1, 'val_f1': 1.0, 'val_loss': 0.0025}, {'epoch': 2, 'val_f1': 1.0, 'val_loss': 0.0004}, {'epoch': 3, 'val_f1': 1.0, 'val_loss..."


## Results

In [4]:
models = ['xlmr', 'afroxlmr']
t = S[S.model.isin(models) & (S.variant == 'clean') & S.set.isin(['test', 'test/tpl', 'chichewa/all'])]
t.pivot_table(index='model', columns='set', values=['f1', 'f1_sd', 'pr_auc']).round(3)

f1                  ...       pr_auc                
set      chichewa/all   test test/tpl  ... chichewa/all   test test/tpl
model                                  ...                             
afroxlmr        0.719  0.776    0.990  ...        0.805  0.958    0.996
xlmr            0.717  0.760    0.983  ...        0.654  0.871    0.991

[2 rows x 9 columns]

In [5]:
BY_SEED[BY_SEED.model.isin(models) & (BY_SEED.variant == 'clean') & (BY_SEED.set == 'test')][['model', 'seed', 'precision', 'recall', 'f1', 'pr_auc']].round(3)

,model,seed,precision,recall,f1,pr_auc
150,afroxlmr,13,1.000,0.667,0.800,0.974
198,afroxlmr,42,1.000,0.617,0.763,0.915
246,afroxlmr,2026,1.000,0.617,0.763,0.986
2143,xlmr,13,1.000,0.617,0.763,0.910
2191,xlmr,42,1.000,0.617,0.763,0.842
2239,xlmr,2026,0.962,0.617,0.752,0.860


**Reading.** Both transformers reach validation F1 1.0 and catch every test scam except the landlord template, so their test F1 is lower than the word-level baselines. Higher capacity did not help on this small, easy dataset: the transformers leaned on the strongest cue (a number in the text) and missed the script without one, although its words (*namba yangu*, *mwenye nyumba*) appear only in training scams. AfroXLMR edges XLM-R on test (0.78 vs 0.76) and ranks better (PR-AUC 0.96 vs 0.87).